In [0]:
# Install libraries
%pip install mistralai==1.2.0 faiss-cpu sentence-transformers

# Restart Python to apply changes
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


creating table for dataset

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Load the dataset
df = spark.read.csv("/Workspace/Users/rohitkushwaha630@gmail.com/AI_practice/practice/dataset.csv", header=True, inferSchema=True)

# Save as Delta table
df.write.format("delta").mode("overwrite").saveAsTable("customer_gold")

In [0]:
from pyspark.sql.functions import concat_ws

# Create the text representation for each row
df_text = df.withColumn(
    "text",
    concat_ws(
        " | ",
        "customer_id",
        "region",
        "plan_type",
        "monthly_usage_gb",
        "churn_score",
        "segment"
    )
)

df_text.select("text").display()

text
101 | Mumbai | prepaid | 3.5 | 0.75 | low
102 | Pune | postpaid | 15.2 | 0.15 | high
103 | Delhi | prepaid | 7.8 | 0.4 | medium
104 | Bangalore | postpaid | 18.5 | 0.1 | high
105 | Chennai | prepaid | 2.1 | 0.85 | low
106 | Hyderabad | prepaid | 6.2 | 0.55 | medium
107 | Kolkata | postpaid | 12.5 | 0.2 | high
108 | Pune | prepaid | 4.0 | 0.65 | low
109 | Mumbai | postpaid | 16.0 | 0.12 | high
110 | Delhi | prepaid | 5.5 | 0.5 | medium


In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Collect texts and generate embeddings
texts = [row["text"] for row in df_text.limit(100).collect()]
embeddings = model.encode(texts)

# Convert to float32 for FAISS compatibility
embeddings = np.array(embeddings).astype('float32')

/local_disk0/.ephemeral_nfs/envs/pythonEnv-a5c1e82b-961e-419d-b750-83d5931a9a0c/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:

import faiss

# Initialize and populate the FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print(f"Indexed {index.ntotal} documents.")

Indexed 10 documents.


In [0]:
import numpy as np

# Define the query
query = "Which customer from Pune has a high churn score?"

# 1. Encode the query using the same model
query_embedding = model.encode([query]).astype('float32')

# 2. Search the FAISS index
k = 3
distances, indices = index.search(query_embedding, k)

# 3. Retrieve the matching text snippets
retrieved_docs = [texts[i] for i in indices[0]]

print(f"Top {k} matches found:")
for doc in retrieved_docs:
    print(f"- {doc}")

Top 3 matches found:
- 102 | Pune | postpaid | 15.2 | 0.15 | high
- 108 | Pune | prepaid | 4.0 | 0.65 | low
- 109 | Mumbai | postpaid | 16.0 | 0.12 | high


In [0]:
from mistralai import Mistral

# Initialize the client
api_key = dbutils.secrets.get(scope="mistral-api", key="api-key")
client = Mistral(api_key=api_key)

def ask_mistral(context, query):
    prompt = f"""
    Context:
    {context}

    Question: {query}

    Answer the question strictly using only the provided context. 
    If the information is not there, state that the context is insufficient.
    """
    
    chat_response = client.chat.complete(
        model="mistral-small-latest",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )

    return chat_response.choices[0].message.content



Test-Model one

In [0]:
#test model-1

responses={}

# Change this line in your notebook to test
query_1 = "Provide details for customers in the 'low' segment with prepaid plans."

# Re-run the chain
context_text = "\n".join(retrieved_docs)
answer_1 = ask_mistral(context_text, query_1)
print(answer_1)

responses[query_1] = answer_1

Customer ID: 108
Location: Pune
Plan Type: prepaid
Usage: 4.0
Spend: 0.65
Segment: low


In [0]:
# test model-2
query_2 = "Which customer from Pune has a high churn score?"

# Re-run the chain
context_text = "\n".join(retrieved_docs)
answer_2 = ask_mistral(context_text, query_2)
print(answer_2)

responses[query_2] = answer_2

Customer with ID 102 from Pune has a high churn score.


In [0]:
# test model -3
query_3 ="Identify the customer with the highest monthly usage in GB from the context provided."
context_text = "\n".join(retrieved_docs)
answer_3 = ask_mistral(context_text, query_3)
print(answer_3)
responses[query_3] = answer_3

The customer with the highest monthly usage in GB is the one with **16.0 GB** (Customer ID: 109, Mumbai, postpaid).


In [0]:



result_df = spark.createDataFrame(responses.items(), ["query", "answer"])

# Append to the output table
result_df.write.mode("append").saveAsTable("llm_outputs")